## make reference for comparable2parallel articles

In [1]:
import os
import pandas as pd
from pathlib import Path
import re


def read_df(fpath, sep = '\t'):
    return pd.read_csv(fpath, sep = sep, index_col = 0)

## check cometkiwi & define threshold

In [3]:

DATA_DIR = "/Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603"
ape_sys_dir = f"{DATA_DIR}/ACL_v4_2026-05/ape_mt/en-fr/bf16_greedy/ape/EuroLLM-22B-Instruct"
mt_sys_dir =  f"{DATA_DIR}/ACL_v4_2026-05/ape_mt/en-fr/bf16_greedy/mt/EuroLLM-22B-Instruct"


comet_fpath1 = f"{ape_sys_dir}/generated.txt.cometkiwi.tsv"
comet_df1 = read_df(comet_fpath1, sep = '\t')

comet_fpath2 = f"{mt_sys_dir}/generated.txt.cometkiwi.tsv"
comet_df2 = read_df(comet_fpath2, sep = '\t')

comet_df1.T

,0,1,2,3,4,5,6,7,8,9,...,6845,6846,6847,6848,6849,6850,6851,6852,6853,6854
0,0.840799,0.858537,0.872305,0.861945,0.781209,0.863348,0.825766,0.868198,0.830575,0.814771,...,0.782117,0.872951,0.867066,0.847254,0.823713,0.862984,0.834431,0.763983,0.721159,0.847493


In [6]:
src_lang = 'en'

res_dict = {}


ape_sys_fpath =  f"{ape_sys_dir}/generated.txt"
ape_sys_sents = open(ape_sys_fpath, 'rt').read().strip().split('\n')

mt_sys_fpath =  f"{mt_sys_dir}/generated.txt"
mt_sys_sents = open(mt_sys_fpath, 'rt').read().strip().split('\n')


src_sep_fpath = f"{DATA_DIR}/ACL_v4_2026-05/extracted_alignments/to_translate.{src_lang}"
src_docs = open(src_sep_fpath, 'rt').read().strip().split('\n')
src_sents = [l.strip() for d in src_docs for l in d.split('<sep>')]




res_dict['src'] = src_sents
res_dict['ape'] = ape_sys_sents
res_dict['cometkiwi-ape'] = comet_df1.T.values[0]
res_dict['mt'] = mt_sys_sents
res_dict['cometkiwi-mt'] = comet_df2.T.values[0]


res_df = pd.DataFrame(res_dict)

In [8]:
res_df[res_df['cometkiwi-ape'] < 0.5]

,src,ape,cometkiwi-ape,mt,cometkiwi-mt
277,3. Word-initial stress,3. Accentuation initiale,0.425273,3. Accentuation initiale des mots,0.534003
310,3.3. Onset lengthening,3.3. Allongement de l'attaque,0.482523,3.3. Allongement de l'attaque,0.482523
358,"Voicing rates, zero-crossing rates and the ali...","Les taux de voisement, de passage par zéro de ...",0.492314,"Les taux de voix, les taux de croisement zéro ...",0.584202
526,"Consequently, no participant emitted sentences...",Il en résulte que les participants n’ont pas f...,0.472966,"Ainsi, aucun participant n’a émis de phrases e...",0.868334
535,The average SNR for the considered sentences o...,"Le meilleur Rapport Signal-Bruit (RSB, en séle...",0.476542,Le rapport signal sur bruit moyen pour les phr...,0.816346
...,...,...,...,...,...
6716,This problem is actually equivalent to countin...,Le nom de la fonction CompterCouplagesParfaits...,0.471624,Ce problème est en fait équivalent au dénombre...,0.789646
6738,But introducing a wildcard on one side of the ...,Si l'on ne peut pas trouver d'alignement de jo...,0.336444,Mais introduire un joker d’un côté de la règle...,0.754837
6751,In lines 20 to 28 we keep only the k-gram quad...,"Aux lignes 20 à 28, seuls les quadruplets de k...",0.452962,"Aux lignes 20 à 28, nous ne gardons que les qu...",0.666450
6782,"For example, the sentences ”An injured woman c...","Par exemple, les phrases « An injured woman co...",0.307132,"Par exemple, les phrases « Une collègue femme ...",0.858511


In [18]:
docid = 'F04-25'

fpath = f"{DATA_DIR}/segmented_aligned_txt_16_cometkiwi/{docid}/align_fr-en.with-cometkiwi.tsv"
d_align_df = pd.read_csv(fpath, sep = '\t', index_col = 0)

comet_df = d_align_df[d_align_df['cometkiwi']>0].sort_values(by = 'cometkiwi')[['fr', 'en', 'cometkiwi']]
comet_df.head(8)

,fr,en,cometkiwi
22,Cette dernière étude [11] utilise une procédur...,The latter study [15] was carried out using pr...,0.509202
45,2.2. Transposition de prosodie : méthode,2.2. Prosody transplantation: methodology,0.644763
32,Les résultats sont fournis section 3 : ils sug...,Results are provided in section 3: they sugges...,0.704908
20,L’Atlas Multimédia Prosodique de l’Espace Roma...,The AMPER [11] linguistic atlas of Romance lan...,0.716725
10,Est-ce un hasard si l’on a choisi le vocable a...,Is it by chance that the term accent has been ...,0.722500
92,Les résultats sur la base de leur seule voix (...,Results on the basis of their sole voices (the...,0.723135
75,Cette tendance en faveur d’un plus grand rôle ...,This tendency in favour of a greater role play...,0.724921
85,Table 2 : Taux d’identification de l’origine d...,Table 2 : Identification scores of the origin ...,0.725576


In [2]:
# Check the quality of reference according to the corresponding range of quality estimation scores

tmp_df = comet_df[comet_df['cometkiwi'] < 0.8]

tmp_df[tmp_df['cometkiwi'] >=0.75 ].values

## make references

- if eval(src, tgt_hum) > 0.8, take TGT_i
- if no translation in TGT_i, take TGT_MT
- if eval(src,tgt_hum) > 0.6 and eval(src, tgt_APE) > 0.6, take best(tgt_APE, TGT_i)
- remaining = tgt_MT

### split by docs

In [4]:
# store cometkiwi, ape, mt by doc


def ape_mt_corpus2docs(sys_list, score_list, docid_list, nb_sent_list, store_dir, key = 'ape'):
    """to_split_list: to split into doc-level data"""
    res_dict = {}
    start = 0
    for docid, size in zip(docid_list, nb_sent_list):
                
        res_dict[key] = sys_list[start:start+size]
        res_dict['cometkiwi'] = score_list[start:start+size]
        
        start += size
        res_df = pd.DataFrame(res_dict)
        
        store_fpath = f"{store_dir}/{docid}/{key}_with-cometkiwi.tsv"
        Path(os.path.dirname(store_fpath)).mkdir(parents = True, exist_ok = True)
        
        res_df.to_csv(store_fpath, sep = '\t' )

DATA_DIR = "/Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603"

version = 'ACL_v4_2026-05'


for direction in ['en-fr', 'fr-en']:
    docid_fpath = f"{DATA_DIR}/{version}/ACL_corpus.{direction}.docid.txt"
    docid_list = [ l.replace('.revised', '') for l in open(docid_fpath, 'rt').read().strip().split('\n')]
    
    src_sep_fpath = f"{DATA_DIR}/{version}/extracted_alignments/to_translate.{direction[:2]}"
    src_docs = open(src_sep_fpath, 'rt').read().strip().split('\n')
    nb_sent_list = [ len(d.split('<sep>')) for d in src_docs ]
    
    for key in ['ape', 'mt']:
        for subfolder in ['bf16_greedy', 'bf16_greedy_term']:
            sys_dir = f"{DATA_DIR}/{version}/ape_mt/{direction}/{subfolder}/{key}/EuroLLM-22B-Instruct"
            store_dir = f"{DATA_DIR}/{version}/ape_mt_by_doc/{direction}/{subfolder}/{key}/EuroLLM-22B-Instruct"
            
            
            sys_sents = open(f"{sys_dir}/generated.txt", 'rt').read().strip().split('\n')
            comet_fpath = f"{sys_dir}/generated.txt.cometkiwi.tsv"
            comet_df = read_df(comet_fpath, sep = '\t')
    
            score_list = comet_df.T.values[0].tolist()
    
            ape_mt_corpus2docs(sys_sents, score_list, docid_list, nb_sent_list, store_dir,  key)
        

In [5]:
fpath = f"{DATA_DIR}/ACL_v4_2026-05/ape_mt_by_doc/fr-en/bf16_greedy_term/mt/EuroLLM-22B-Instruct/F04-46/mt_with-cometkiwi.tsv"
read_df(fpath)

,mt,cometkiwi
0,A new approach to language modeling by dynamic...,0.872714
1,1. INTRODUCTION,0.824204
2,The role of a statistical language model is to...,0.877065
3,This modeling is ensured by assigning a probab...,0.859357
4,This probability is estimated using a Markov c...,0.854551
...,...,...
105,"In this article, we chose to test several type...",0.789988
106,The results show that it is promising and dese...,0.845714
107,"Obviously, other experiments of the same kind ...",0.865915
108,But our main goal is to develop an algorithm t...,0.807523


### make refs

In [6]:



def mask_equal_src_mt(hum_df, mt_df, src_lang):
    stat_df1 = pd.DataFrame(index=hum_df.index)    
    stat_df1['mt=src'] = [0]*len(hum_df)
    
    mask = hum_df[src_lang].apply(lambda x: x.lower()) == \
       mt_df['mt'].apply(lambda x: x.lower())
    stat_df1.mask(mask, -1)
    return stat_df1

def prepare_references_file(hum_df, ape_df, mt_df, src_lang, tgt_lang, thred1 = 0.8, thred2 = 0.6 ):
    
    stat_df = pd.DataFrame(index=hum_df.index)
    stat_df['type'] = [0]*len(hum_df)

    # for cometkiwi(src, MT) == 1 : mark as -1 
    stat_df1 = mask_equal_src_mt(hum_df, mt_df, src_lang)

    # make reference : 
    # 1. empty = MT
    mt_empty_df = mt_df[hum_df[tgt_lang].isna()]['mt']
    
    hum_df = hum_df.dropna()
    ape_df = ape_df.loc[hum_df.index]
    mt_df = mt_df.loc[hum_df.index]
    
    # 2. for cometkiwi(src, APE) < 0.6 and cometkiwi(src, Human) < 0.6, take MT
    mt_df = mt_df.loc[(hum_df['cometkiwi'] < thred2 ) & (ape_df['cometkiwi'] < thred2 )]['mt']
    hum_df = hum_df[~hum_df.index.isin(mt_df.index)]
    ape_df = ape_df[~ape_df.index.isin(mt_df.index)]
    
    # 3. & 4. cometkiwi > 0.8 = keep origin, others = argmax(cometkiwi(src, APE), cometkiwi(src, human) )
    # for cometkiwi(src, Human) >0.8, take hum, 
    diff_df = hum_df['cometkiwi'] - ape_df['cometkiwi']
    diff_df[hum_df['cometkiwi'] > thred1] = 1
    ape_df = ape_df[diff_df < 0]['ape']
        
    ref_df = pd.concat( [
        mt_empty_df,
        mt_df,
        ape_df,    
        hum_df[diff_df >= 0][tgt_lang],  
    ]).to_frame()
    
    # stat 0 = origin, 1 = APE, 2 = MT
    stat_df.loc[mt_empty_df.index] = 2
    stat_df.loc[mt_df.index] = 2
    stat_df.loc[ape_df.index] = 1

    stat_dict = {
        0: len(stat_df[stat_df==0] ),
        1: len(ape_df),
        2: len(mt_empty_df) + len(mt_df),
    }
    ref_sents = ref_df.sort_index().T.values[0]

    return ref_sents, stat_dict, stat_df, stat_df1


def make_dataset_cometkiwi(
    docid_list,
    data_dir,
    root_store_dir, 
    get_data_fpath_func, 
    get_store_fpath_func, 
    src_lang = 'en', 
    tgt_lang = 'fr',
    ctx_type = 'pmi',
    thred1 = 0.8, 
    thred2 = 0.6
):
    assert(tgt_lang in ['en', 'fr'])
    
    stat_dict = {}
    for d in docid_list:
                
        fpath = get_data_fpath_func('hum', data_dir, src_lang, tgt_lang, d)

        hum_df = pd.read_csv(fpath, sep = '\t', index_col = 0)
        hum_df = hum_df.loc[hum_df[ src_lang ].dropna().index].reset_index()
        
        fpath = get_data_fpath_func('mt', data_dir, src_lang, tgt_lang, d, ctx_type = ctx_type)
        mt_df = pd.read_csv(fpath, sep = '\t', index_col = 0)
        
        fpath = get_data_fpath_func('ape', data_dir, src_lang, tgt_lang, d,  ctx_type = ctx_type)
        ape_df = pd.read_csv(fpath, sep = '\t', index_col = 0)
        
        assert(len(hum_df) == len(mt_df))
        assert(len(hum_df) == len(ape_df))
        
        # make references
        ref_sents, f_stat_dict, stat_df, stat_df1 = prepare_references_file(
            hum_df, 
            ape_df, 
            mt_df, 
            src_lang, 
            tgt_lang, 
            thred1 = thred1, 
            thred2 = thred2,
        )
            
        assert(len(mt_df) == len(ref_sents))
        
        stat_dict[d] = f_stat_dict
    
        # store
        store_fpath= get_store_fpath_func('tgt', root_store_dir, src_lang, tgt_lang, d)
        Path(os.path.dirname(store_fpath)).mkdir(parents=True, exist_ok=True)
        with open(store_fpath, 'w', encoding = 'utf-8') as f:
            f.write('\n'.join(ref_sents))

        pd.concat([stat_df, stat_df1], axis = 1).to_csv(f"{store_fpath}.stat.tsv", sep = '\t')
        # stat_df.to_csv(f"{store_fpath}.stat.tsv", sep = '\t')
        
        # src
        os.system(f"cp  {get_data_fpath_func('src', data_dir, src_lang, tgt_lang, d)}  {get_store_fpath_func('src', root_store_dir, src_lang, tgt_lang, d)}")

    all_stat_df = pd.DataFrame(stat_dict).T
    all_stat_df.to_csv(get_store_fpath_func('stat', root_store_dir, src_lang, tgt_lang), sep = '\t')

    to_print_stat = all_stat_df.describe().T
    to_print_stat.to_csv(get_store_fpath_func('describe_stat', root_store_dir, src_lang, tgt_lang), sep = '\t')
    print(to_print_stat[['count','mean','min','max']].round(0).style.format(precision = 0).to_latex())
    print(to_print_stat[['count','mean','min','max']].round(0).to_markdown())


def get_acl_data_fpath(key, data_dir, src, tgt, d, ctx_type = 'pmi'):
    fname_prefix = 'acl_corpus'
    version = 'ACL_v4_2026-05'    

        
    # intermediate files
    if key == 'hum':
        return f"{data_dir}/segmented_aligned_txt_16_cometkiwi/{d}/align_fr-en.with-cometkiwi.tsv"
            
    if key == 'src':
        src_lang = 'English' if src == 'en' else 'French'
        return f"{data_dir}/{version}/extracted_alignments/{d}/non_empty_{src_lang}/alignments.{src}"

    if key in [ 'ape', 'mt']:
        if ctx_type == 'pmi':
            subfolder = 'bf16_greedy'
        elif  ctx_type == 'pmi_term':
            subfolder = 'bf16_greedy_term'
        else:
            print('unknown ctx type')
        return f"{data_dir}/{version}/ape_mt_by_doc/{src}-{tgt}/{subfolder}/{key}/EuroLLM-22B-Instruct/{d}/{key}_with-cometkiwi.tsv"
        

    
def get_acl_store_fpath(key, root_store_dir, src, tgt, d = None):
    # f"{root_store_dir}/parallel_corpus_cometkiwi
    fname_prefix = 'acl_corpus'
    if key in [ 'tgt', 'src']:
        assert(d is not None)
    
    # to store
    if key == 'tgt':
        return  f"{root_store_dir}/parallel_corpus/{src}_{tgt}/{d}/{fname_prefix}.{tgt}.txt"
    if key == 'src':
        return f"{root_store_dir}/parallel_corpus/{src}_{tgt}/{d}/{fname_prefix}.{src}.txt"
        
    if key == 'stat':
        return f"{root_store_dir}/parallel_corpus/{fname_prefix}_{src}_{tgt}.stat.tsv"
    if key == 'describe_stat':
        return f"{root_store_dir}/parallel_corpus/describe_{fname_prefix}_{src}_{tgt}.stat.tsv"
    

In [109]:




src_lang = 'en'
tgt_lang = 'fr'


thred1 = 0.8
thred2 = 0.6

for ctx_type in ['pmi',  'pmi_term']:

    src_dir = f"{DATA_DIR}/segmented_aligned_txt_16_cometkiwi"
    docid_list = [ d for d in os.listdir(src_dir) if  os.path.isdir(f"{src_dir}/{d}") and d[0] != '.']
    

    store_dir = f"{DATA_DIR}/ACL_v4_2026-05/corpus_ACL_v4_2026-05/cometkiwi_{ctx_type}"
    make_dataset_cometkiwi(
        docid_list,
        DATA_DIR,
        store_dir, 
        get_acl_data_fpath, 
        get_acl_store_fpath, 
        src_lang = src_lang, 
        tgt_lang = tgt_lang, 
        ctx_type = ctx_type, 
        thred1 = thred1, 
        thred2 = thred2
    )

\begin{tabular}{lrrrr}
 & count & mean & min & max \\
0 & 35 & 196 & 107 & 477 \\
1 & 35 & 81 & 26 & 210 \\
2 & 35 & 44 & 7 & 202 \\
\end{tabular}

|    |   count |   mean |   min |   max |
|---:|--------:|-------:|------:|------:|
|  0 |      35 |    196 |   107 |   477 |
|  1 |      35 |     81 |    26 |   210 |
|  2 |      35 |     44 |     7 |   202 |
\begin{tabular}{lrrrr}
 & count & mean & min & max \\
0 & 35 & 196 & 107 & 477 \\
1 & 35 & 82 & 24 & 207 \\
2 & 35 & 43 & 7 & 207 \\
\end{tabular}

|    |   count |   mean |   min |   max |
|---:|--------:|-------:|------:|------:|
|  0 |      35 |    196 |   107 |   477 |
|  1 |      35 |     82 |    24 |   207 |
|  2 |      35 |     43 |     7 |   207 |


In [8]:
src_lang = 'fr'
tgt_lang = 'en'


thred1 = 0.8
thred2 = 0.6

for ctx_type in ['pmi',  'pmi_term']:

    src_dir = f"{DATA_DIR}/segmented_aligned_txt_16_cometkiwi"
    docid_list = [ d for d in os.listdir(src_dir) if  os.path.isdir(f"{src_dir}/{d}") and d[0] != '.']
    

    store_dir = f"{DATA_DIR}/ACL_v4_2026-05/corpus_ACL_v4_2026-05/cometkiwi_{ctx_type}"
    make_dataset_cometkiwi(
        docid_list,
        DATA_DIR,
        store_dir, 
        get_acl_data_fpath, 
        get_acl_store_fpath, 
        src_lang = src_lang, 
        tgt_lang = tgt_lang, 
        ctx_type = ctx_type, 
        thred1 = thred1, 
        thred2 = thred2
    )

\begin{tabular}{lrrrr}
 & count & mean & min & max \\
0 & 35 & 192 & 103 & 408 \\
1 & 35 & 74 & 17 & 175 \\
2 & 35 & 42 & 4 & 221 \\
\end{tabular}

|    |   count |   mean |   min |   max |
|---:|--------:|-------:|------:|------:|
|  0 |      35 |    192 |   103 |   408 |
|  1 |      35 |     74 |    17 |   175 |
|  2 |      35 |     42 |     4 |   221 |
\begin{tabular}{lrrrr}
 & count & mean & min & max \\
0 & 35 & 192 & 103 & 408 \\
1 & 35 & 75 & 19 & 183 \\
2 & 35 & 41 & 3 & 209 \\
\end{tabular}

|    |   count |   mean |   min |   max |
|---:|--------:|-------:|------:|------:|
|  0 |      35 |    192 |   103 |   408 |
|  1 |      35 |     75 |    19 |   183 |
|  2 |      35 |     41 |     3 |   209 |
